In [1]:
# Fase 0: Preparación del entorno
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
import time

# Configuración estética de gráficas
plt.style.use('seaborn-v0_8-darkgrid')

In [2]:
# Fase 1 y 2: Sistema de 3 nodos (verificación manual y computacional)

# Matriz K y vector F para 3 nodos (según enunciado)
K_3 = np.array([
    [2.0, -1.0, 0.0],
    [-1.0, 2.0, -1.0],
    [0.0, -1.0, 1.0]
], dtype=float)

F_3 = np.array([10.0, 0.0, 5.0], dtype=float)

# Solución numérica directa (fase 2)
u_3 = np.linalg.solve(K_3, F_3)
print("Desplazamientos u_3 (solución directa):", u_3)

# Si quisieras verificar el paso de eliminación Gaussiana a mano,
# podrías implementar una pequeña función de eliminación para el primer
# multiplicador L21. No es necesario para la evaluación final, pero útil para aprendizaje.
# Ejemplo (solo observación; NO es parte de la solución exigida):
def primer_paso_eliminacion_gauss(K):
    K = K.astype(float).copy()
    # El multipicador para eliminar K[1,0] usando fila 0 es l21 = K[1,0]/K[0,0]
    l21 = K[1,0] / K[0,0]
    # Actualizar fila 1: fila1 := fila1 - l21 * fila0
    K[1, :] = K[1, :] - l21 * K[0, :]
    return l21, K

l21, K_updated = primer_paso_eliminacion_gauss(K_3)
print("Primer multiplicador L21 (observación):", l21)
print("Matriz K tras el primer paso de eliminación Gaussiana (observación):\n", K_updated)

# Nota: La solución exacta debe coincidir con u_3, y lo anterior sirve solo como verificación manual.


Desplazamientos u_3 (solución directa): [15. 20. 25.]
Primer multiplicador L21 (observación): -0.5
Matriz K tras el primer paso de eliminación Gaussiana (observación):
 [[ 2.  -1.   0. ]
 [ 0.   1.5 -1. ]
 [ 0.  -1.   1. ]]


In [ ]:
# Fase 3: Análisis paramétrico (5000 nodos) con dos enfoques

N = 5000
# Construcción de la gran estructura (tridiagonal)
K_masiva = 2 * np.eye(N) - 1 * np.eye(N, k=1) - 1 * np.eye(N, k=-1)
# Frontera libre en el extremo derecho
K_masiva[-1, -1] = 1

# Generación de 1000 cargas aleatorias
np.random.seed(42)
num_casos = 1000
cargas_viento = np.random.rand(num_casos, N) * 100  # cada fila es un vector b

# Enfoque A: la Fuerza Bruta (Inversa dentro del bucle) - para comparación
start_A = time.time()
u_A = np.empty_like(cargas_viento)
# Nota: invertir dentro del bucle es muy costoso; aquí se ilustra literalmente
for i in range(num_casos):
    b = cargas_viento[i]
    K_inv = np.linalg.inv(K_masiva)  # inversión dentro del bucle (costoso)
    u_A[i, :] = K_inv @ b
time_A = time.time() - start_A
print("Tiempo Enfoque A (inversa dentro del bucle):", time_A, "segundos")

# Enfoque B: LU pre-factorizado (recomendado)
start_B = time.time()
LU_y_pivotes = la.lu_factor(K_masiva)  # factorizar fuera del bucle
u_B = np.empty_like(cargas_viento)

for i in range(num_casos):
    b = cargas_viento[i]
    u_B[i, :] = la.lu_solve(LU_y_pivotes, b)
time_B = time.time() - start_B
print("Tiempo Enfoque B (LU pre-factorizado):", time_B, "segundos")


In [ ]:
# Fase 4: Análisis visual del peor caso (Enfoque B)
# Identificar el caso con mayor desplazamiento en el último nodo
max_last_node_idx = np.argmax(u_B[:, -1])
max_desplazamiento = u_B[max_last_node_idx, -1]
print("Caso crítico (índice):", max_last_node_idx)
print("Desplazamiento máximo en el último nodo:", max_desplazamiento)

# Gráfico de perfil de desplazamientos para ese caso crítico
critico_u = u_B[max_last_node_idx, :]
plt.figure(figsize=(10,4))
plt.plot(range(1, N+1), critico_u, marker='o', linestyle='-', markersize=2)
plt.xlabel('Nodo')
plt.ylabel('Desplazamiento u_i')
plt.title('Perfil de desplazamientos para el caso crítico (último nodo con mayor |u|)')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Verificación adicional (opcional)
# Desplazamiento en el primer nodo (empotrado a la pared) para verificar coherencia física
print("Desplazamiento en el primer nodo (u<a href="" class="citation-link" target="_blank" style="vertical-align: super; font-size: 0.8em; margin-left: 3px;">[0]</a>) :", u_B[0, 0])

# Si quisieras, puedes comprobar que el último nodo no está forzado a cero necesariamente
# debido a la frontera libre (según la configuración del enunciado).
